In [1]:
import numpy as np
import torch
import torch.nn as nn

from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

In [2]:
data = np.load("pems04.npz")

traffic = data["data"][:,:,2]

print(traffic.shape)

(16992, 307)


In [3]:
scaler = MinMaxScaler()

traffic_scaled = scaler.fit_transform(
    traffic
)

In [4]:
X = []
y = []

sequence_length = 12

for i in range(
    len(traffic_scaled)
    - sequence_length
):

    X.append(
        traffic_scaled[
            i:i+sequence_length
        ]
    )

    y.append(
        traffic_scaled[
            i+sequence_length
        ]
    )

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

(16980, 12, 307)
(16980, 307)


In [5]:
split = int(
    len(X) * 0.8
)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

In [6]:
X_train = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_test = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_train = torch.tensor(
    y_train,
    dtype=torch.float32
)

y_test = torch.tensor(
    y_test,
    dtype=torch.float32
)

In [7]:
train_dataset = TensorDataset(
    X_train,
    y_train
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

In [8]:
import torch
import torch.nn as nn

class GRUModel(nn.Module):

    def __init__(
        self,
        num_nodes=307,
        hidden_size=64
    ):

        super().__init__()

        self.gru = nn.GRU(
            input_size=num_nodes,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_size,
            num_nodes
        )

    def forward(self,x):

        output, hidden = self.gru(x)

        hidden = hidden[-1]

        x = self.fc(hidden)

        return x

In [9]:
model = GRUModel()

X_batch, y_batch = next(
    iter(train_loader)
)

pred = model(X_batch)

print(pred.shape)
print(y_batch.shape)

torch.Size([64, 307])
torch.Size([64, 307])


In [10]:
model = GRUModel()

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 30

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        pred = model(X_batch)

        loss = criterion(
            pred,
            y_batch
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{epochs} "
        f"Loss: {total_loss/len(train_loader):.6f}"
    )

Epoch 1/30 Loss: 0.046836
Epoch 2/30 Loss: 0.009400
Epoch 3/30 Loss: 0.007754
Epoch 4/30 Loss: 0.007144
Epoch 5/30 Loss: 0.006724
Epoch 6/30 Loss: 0.006343
Epoch 7/30 Loss: 0.005974
Epoch 8/30 Loss: 0.005650
Epoch 9/30 Loss: 0.005397
Epoch 10/30 Loss: 0.005206
Epoch 11/30 Loss: 0.005017
Epoch 12/30 Loss: 0.004871
Epoch 13/30 Loss: 0.004733
Epoch 14/30 Loss: 0.004596
Epoch 15/30 Loss: 0.004479
Epoch 16/30 Loss: 0.004370
Epoch 17/30 Loss: 0.004261
Epoch 18/30 Loss: 0.004159
Epoch 19/30 Loss: 0.004052
Epoch 20/30 Loss: 0.003965
Epoch 21/30 Loss: 0.003873
Epoch 22/30 Loss: 0.003791
Epoch 23/30 Loss: 0.003704
Epoch 24/30 Loss: 0.003637
Epoch 25/30 Loss: 0.003567
Epoch 26/30 Loss: 0.003500
Epoch 27/30 Loss: 0.003438
Epoch 28/30 Loss: 0.003381
Epoch 29/30 Loss: 0.003331
Epoch 30/30 Loss: 0.003284


In [11]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
import numpy as np


test_dataset = TensorDataset(
    X_test,
    y_test
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

all_predictions = []
all_targets = []

model.eval()

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        pred = model(X_batch)

        all_predictions.append(
            pred.numpy()
        )

        all_targets.append(
            y_batch.numpy()
        )

predictions = np.concatenate(
    all_predictions,    
    axis=0
)

true_values = np.concatenate(
    all_targets,
    axis=0
)

mae = mean_absolute_error(
    true_values,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        true_values,
        predictions
    )
)

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 0.04078451544046402
RMSE: 0.06828886210088475
